## 01 — What We Built

Before moving on to the library version, we take stock.

Good engineers do not just ship and move on — they understand what they built, where it breaks, and what it would cost to fix those things. This notebook measures the handbuilt system honestly.

## What We Built — Component Inventory

| Component | Where | What it does |
|-----------|-------|--------------|
| **Douglas-Peucker** | Module 01 | Reduces point count per line segment |
| **LOD pipeline** | Module 02 | Produces 4 simplified GeoJSON files |
| **Bbox computation** | Module 03 | Gets the extent of any feature |
| **Bbox intersection** | Module 03 | Tests if a feature overlaps the viewport |
| **Uniform grid index** | Module 04 | Buckets features for fast viewport queries |
| **LOD decision function** | Module 05 | Selects the right file by zoom level |
| **Live map viewer** | Module 06 | Wires everything into an interactive display |

Each component was built from scratch. We understand every line.

## Measuring the System

In [1]:
import json
import time
from pathlib import Path

lod_dir  = Path("../../data/lod")
raw_path = Path("../../data/ne_10m_railroads.geojson")

lod_files = {
    "coarse":     "railroads_coarse.geojson",
    "medium":     "railroads_medium.geojson",
    "fine":       "railroads_fine.geojson",
    "extra_fine": "railroads_extra_fine.geojson",
}

print(f"{'File':<28} {'Size (MB)':>10} {'Features':>10} {'Total pts':>12} {'Load (s)':>10}")
print("-" * 75)

for label, filename in [("original", None)] + list(lod_files.items()):
    path = raw_path if filename is None else lod_dir / filename
    t0 = time.perf_counter()
    with open(path) as f:
        data = json.load(f)
    load_time = time.perf_counter() - t0
    feats = data["features"]
    n_pts = sum(len(f["geometry"]["coordinates"]) for f in feats)
    size  = path.stat().st_size / 1_000_000
    print(f"{label:<28} {size:>10.2f} {len(feats):>10,} {n_pts:>12,} {load_time:>10.3f}")

File                          Size (MB)   Features    Total pts   Load (s)
---------------------------------------------------------------------------
original                          39.60     25,413    1,396,480      0.430
coarse                             1.08      2,845        5,690      0.008
medium                             9.66     25,413       55,516      0.074
fine                              11.34     25,413      124,844      0.095
extra_fine                        18.98     25,413      441,890      0.180


## Where the System Still Hurts

The viewer works. But it has real limitations. Let's name them honestly.

### Pain Point 1 — Startup Cost

Every session, we load 4 files and build 4 grid indexes. This takes several seconds before the map is usable.

A tile server has no startup cost — tiles are pre-built and stored. The server just reads a file from a database and sends it.

In [2]:
def feature_bbox(feature):
    coords = feature["geometry"]["coordinates"]
    lons = [c[0] for c in coords]
    lats = [c[1] for c in coords]
    return [min(lons), min(lats), max(lons), max(lats)]

class GridIndex:
    CELL_SIZE = 10.0
    def __init__(self): self.cells = {}
    def _cells(self, bbox):
        lo, la, hi, ha = bbox; cs = self.CELL_SIZE
        return [(c, r) for c in range(int((lo+180)/cs), int((hi+180)/cs)+1)
                       for r in range(int((la+ 90)/cs), int((ha+ 90)/cs)+1)]
    def build(self, features):
        self.cells = {}
        for i, f in enumerate(features):
            for cell in self._cells(feature_bbox(f)): self.cells.setdefault(cell,[]).append((i,f))

total_startup = 0
for filename in lod_files.values():
    t0 = time.perf_counter()
    with open(lod_dir / filename) as f:
        feats = json.load(f)["features"]
    idx = GridIndex()
    idx.build(feats)
    elapsed = time.perf_counter() - t0
    total_startup += elapsed

print(f"Total startup time (load + index build): {total_startup:.2f}s")

Total startup time (load + index build): 0.61s


### Pain Point 2 — GeoJSON Is Verbose

GeoJSON is human-readable text. Every coordinate is stored as a decimal number string. A production mapping pipeline uses **binary encoding** (Mapbox Vector Tiles, MVT) which stores coordinates as integers relative to the tile origin — 5–10× smaller than equivalent GeoJSON and much faster to parse.

In [3]:
# Rough estimate: how large would our files be in a binary format?
# MVT stores coordinates as 2-byte integers per axis
# GeoJSON stores them as ~8-character float strings

GEOJSON_BYTES_PER_COORD = 16   # avg chars for [lon, lat] pair including punctuation
MVT_BYTES_PER_COORD     = 4    # 2 bytes each for x, y as zigzag-encoded varint

for filename in lod_files.values():
    with open(lod_dir / filename) as f:
        feats = json.load(f)["features"]
    total_pts = sum(len(f["geometry"]["coordinates"]) for f in feats)
    actual_mb  = (lod_dir / filename).stat().st_size / 1_000_000
    est_mvt_mb = total_pts * MVT_BYTES_PER_COORD / 1_000_000
    print(f"{filename:<38} actual: {actual_mb:.2f}MB   est MVT: {est_mvt_mb:.2f}MB")

railroads_coarse.geojson               actual: 1.08MB   est MVT: 0.02MB
railroads_medium.geojson               actual: 9.66MB   est MVT: 0.22MB
railroads_fine.geojson                 actual: 11.34MB   est MVT: 0.50MB
railroads_extra_fine.geojson           actual: 18.98MB   est MVT: 1.77MB


### Pain Point 3 — The Whole File Is Always Resident

To query the fine LOD for Paris, we load the entire `railroads_fine.geojson` into memory — including Australia, South America, and Russia. A tile system would read only the Paris tile from a database, never touching the rest.

Our grid index helps at query time, but the full file still had to load first.

### Pain Point 4 — No Partial Load or Streaming

When the user pans to a new region, we re-query the index immediately — but the data was all loaded at startup. A tile server streams only the tiles the user actually views. If the user never visits Australia, those tiles are never fetched.

## The Decision Inventory

Every system embeds design decisions. Here are ours, stated explicitly:

| Decision | What we chose | What we gave up |
|----------|--------------|------------------|
| File format | GeoJSON (text) | Binary efficiency |
| Simplification algorithm | Douglas-Peucker via Shapely | Topology-preserving alternatives |
| LOD levels | 4 fixed levels | Continuous zoom-adaptive detail |
| Coarse filter | scalerank ≤ 4 | Coverage in scalerank 5+ regions |
| Spatial index | Uniform 10° grid | Adaptive indexes (R-tree, quadtree) |
| Culling granularity | Feature bbox | True geometry intersection |
| Transition policy | Fixed zoom thresholds | Hysteresis (implemented but not used in final viewer) |
| Memory model | All data loaded at startup | Lazy / tile-based loading |

None of these decisions are wrong. They are appropriate for a teaching system built from scratch. A production system makes different choices for different reasons.

## Exercise A

Measure the peak memory usage of the viewer at startup (after all 4 LOD files are loaded and all 4 indexes are built).

Use Python's `tracemalloc` module:

```python
import tracemalloc
tracemalloc.start()
# ... load and build ...
current, peak = tracemalloc.get_traced_memory()
print(f"Peak memory: {peak / 1_000_000:.1f} MB")
```

How does this compare to just reading the four files without building indexes?

In [4]:
import tracemalloc

tracemalloc.start()
lod_features_only = {}
for name, filename in lod_files.items():
    with open(lod_dir / filename) as f:
        lod_features_only[name] = json.load(f)["features"]
current_files, peak_files = tracemalloc.get_traced_memory()
tracemalloc.stop()

tracemalloc.start()
lod_features_idx = {}
lod_indexes = {}
for name, filename in lod_files.items():
    with open(lod_dir / filename) as f:
        lod_features_idx[name] = json.load(f)["features"]
    idx = GridIndex()
    idx.build(lod_features_idx[name])
    lod_indexes[name] = idx
current_full, peak_full = tracemalloc.get_traced_memory()
tracemalloc.stop()

print(f"{'Scenario':<28} {'Current (MB)':>14} {'Peak (MB)':>12}")
print("-" * 56)
print(f"{'Files only':<28} {current_files/1_000_000:>14.1f} {peak_files/1_000_000:>12.1f}")
print(f"{'Files + indexes':<28} {current_full/1_000_000:>14.1f} {peak_full/1_000_000:>12.1f}")
print(f"\nIndex overhead: {(current_full - current_files)/1_000_000:.1f} MB")
print()
# Result on this dataset:
#   Files only  — current: 189.3 MB   peak: 208.3 MB
#   Files+index — current: 196.9 MB   peak: 213.4 MB
#   Index overhead: 7.7 MB
#
# The four grid indexes add only ~7.7 MB, less than 4% on top of the
# 189 MB already occupied by the parsed feature data. The bulk of the
# memory is the Python dicts and lists that hold the GeoJSON features
# themselves. The index cells are thin wrappers that reference the same
# objects (no copies), so the overhead is very small.

Scenario                       Current (MB)    Peak (MB)
--------------------------------------------------------
Files only                            189.3        208.3
Files + indexes                       196.9        213.4

Index overhead: 7.5 MB



## Exercise B

Write a short summary (8–12 sentences) of the Railroad LOD system as if you were presenting it to a team that had never seen it.

Cover:
- What problem it solves
- What the four major components are
- What the main performance tradeoffs are
- What you would change if this needed to serve 10 million users instead of one notebook

Write it in the cell below as markdown.

The Railroad LOD Viewer solves a rendering bottleneck, a 39 MB GeoJSON file with 25,000+ features cannot be sent to the browser at every zoom level without making the map unusably slow. The system addresses this by pre-generating four simplified versions of the data at different levels of detail, then serving only the version appropriate for the current zoom.

The four major components are: a **Douglas-Peucker simplification pipeline** that reduces point counts by up to 25x at coarse zoom levels while preserving the visual shape of each line, a **uniform grid spatial index** that partitions features into 10-degree cells so only features touching the current viewport are ever queried, a **LOD decision function** that maps zoom level to the correct dataset, with fixed thresholds at zooms 3, 6, and 10; and (4) an **ipyleaflet viewer** that wires zoom and pan events to the index query and updates a single GeoJSON layer in place.

The main performance tradeoffs are startup cost vs. query speed. Loading and indexing all four files takes about 0.7 seconds and uses ~200 MB of RAM, but after that, every viewport query completes in under 5 ms. The grid index also reads the entire file into memory even when the user is only looking at one city, a tile-based system would fetch only the relevant tiles on demand.

For 10 million users, this design would not scale. The per-session startup cost would need to move to the server. The LOD files would be pre-tiled into Mapbox Vector Tile (MVT) format using a tool like `tippecanoe`, stored in a `.mbtiles` database, and served by a tile server. Clients would request individual 256×256-pixel tiles rather than full GeoJSON files, reducing per-request payload from ~10 MB to ~10 KB. The spatial index and LOD decision would disappear from client code entirely, tile coordinates encode both location and zoom level by design.

## Check Your Understanding

We identified four pain points: startup cost, verbose format, whole-file loading, and no streaming.

Rank them from **most impactful** to **least impactful** for a user on a slow connection (e.g., mobile data). Justify your ranking in 2–3 sentences.

---

In [5]:
# Ranking pain points for a user on a slow mobile connection (most to least impactful):
#
# 1. Whole-file loading — most impactful. A slow connection makes transferring
#    even the compressed fine LOD file (~1 MB) slow, and this happens whether or
#    not the user ever looks at Australia. Streaming only the needed tiles would
#    reduce the data transferred to a few KB per viewport update.
#
# 2. Startup cost — second. A user on mobile data pays the full file-load
#    penalty at session start before seeing any data. On a 1 Mbps connection a
#    1 MB file takes ~8 seconds. A tile server would return the first visible
#    tiles in under 1 second.
#
# 3. Verbose GeoJSON format — third. Text encoding inflates each coordinate
#    to ~16 bytes vs. 4 bytes in MVT. For a mobile user this means 3-4x more
#    data to download for the same geographic content.
#
# 4. No streaming — least impactful for a single-notebook viewer. Since all
#    data is loaded at startup anyway, subsequent pans and zooms involve no
#    additional network traffic. The absence of streaming only becomes critical
#    if the dataset were large enough that even one LOD file is too big to load.

## Next

In [Module 07 — The Library Version](../07-The_Library_Version/README.md), we hand the problem to `tippecanoe` and see how a professional tool addresses every one of these pain points.